# AF Kenya: PopClusters Community Data - Compare to Ghana

- Explore PopClusters data (pre-clustered, Kenya-specific)
- See how communities look
- Overlay Aquaya WPs - generate WP and community stats (representative?)


# Setup

In [ ]:
!pip install rasterio folium matplotlib mapclassify

In [ ]:
import rasterio
import pandas as pd
import geopandas as gpd
import plotly.express as px

In [ ]:
# Sample filepath / load: pd.read_csv("drive/MyDrive/data.csv")
from google.colab import drive
drive.mount('/content/drive')
data_dir = "drive/MyDrive/Colab Notebooks/Data/dd-afkenya/"
data_dir

# Config & Load Data

In [ ]:
# Projected CRS (enabling lat/lon distance comparisons in human units eg: KM)
PROJ_CRS = "EPSG:2136"

# Non-projected general CRS
GNRL_CRS = "EPSG:4326"

In [ ]:
sample_adms = [
    "Ahafo",
    "Bono"
]

## ADM Boundaries (UN - COD)


In [ ]:
adm_file = data_dir + "gha_admbnda_gss_20210308_SHP"
gpd.list_layers(adm_file)

In [ ]:

adm_df = gpd.read_file(adm_file, layer="gha_admbnda_adm1_gss_20210308").to_crs(crs=PROJ_CRS)
sample_adm_df = adm_df[adm_df["ADM1_EN"].isin(sample_adms)].copy()

adm_df.shape, sample_adm_df.shape

## Populated Places (HOTOSM)

In [ ]:
# HOTOSM Populated Places
hotosm_file = data_dir + "hotosm_gha_populated_places_points_shp.zip"
hotosm_df = gpd.read_file(hotosm_file).to_crs(crs=PROJ_CRS)

# Drop isolated dwellings
# hotosm_df = hotosm_df[~hotosm_df["place"].isin(["isolated_dwelling"])]

# Sample by intersection w/Adm. boundaries
sample_hotosm_df = hotosm_df[hotosm_df.apply(lambda r: r["geometry"].intersects(sample_adm_df.geometry).sum() > 0, axis=1)].copy()

hotosm_df.shape, sample_hotosm_df.shape

## **Aquaya WPs (Systems, Labs)**

In [ ]:
# Waterpoints
wp_file = data_dir + "AF Kenya - Consolidated Water Systems.xlsx"
wp_df = pd.read_excel(wp_file, sheet_name="Systems")
wp_df = gpd.GeoDataFrame(wp_df, geometry=gpd.points_from_xy(wp_df["Longitude"], wp_df["Latitude"], crs=GNRL_CRS)).to_crs(crs=PROJ_CRS)
# Counties mis-labeled in sheet; rely on actual geographical intersection
# sample_wp_df = wp_df[wp_df["County"].isin(sample_counties)].copy()
sample_wp_df = wp_df[wp_df.apply(lambda r: r["geometry"].intersects(sample_adm_df.geometry).sum() > 0, axis=1)].copy()

wp_df.shape, sample_wp_df.shape

In [ ]:
# Labs
labs_df = pd.read_excel(wp_file, sheet_name="Labs")
labs_df = gpd.GeoDataFrame(labs_df, geometry=gpd.points_from_xy(labs_df["Longitude"], labs_df["Latitude"], crs=GNRL_CRS)).to_crs(crs=PROJ_CRS)
# Counties mis-labeled in sheet; rely on actual geographical intersection
sample_labs_df = labs_df[labs_df.apply(lambda r: r["geometry"].intersects(sample_adm_df.geometry).sum() > 0, axis=1)].copy()

labs_df.shape, sample_labs_df.shape

## **PopCluster - Ghana**

Publication's population clusters (custom alg. derived specifically for Sub-Saharan Africa c. 2020 with all kinds of datasets - see notes and publication).

Population clusters for sub-Saharan Africa. The population clusters include the following data:
```
1.	id – The IDs are given as a unique number for each cluster
2.  Country - Name of the country.
3.	Population – This is the population in each cluster obtained from the population dataset (GHSL for Somalia, Sudan and South Sudan and HRSL for the rest of the countries). The population in these clusters are calibrated to 2016 population values
4.	NightLight – This value is obtained from the night-time light map and represents the maximum luminance detected in each cluster based on the 2016 stable light product available.
5.	ElecPop – The number of people in each cluster who live in areas in which the stable light product detect light sources.
6.	Area – The area of each cluster given in square kilometrers.
7.	IsUrban – Classifies areas as either urban (2), peri-urban (1) or rural (0).
```

In [ ]:
popclusters_file = data_dir + "PopClusters - Ghana"
popclusters_df = gpd.read_file(popclusters_file).to_crs(crs=PROJ_CRS)

# Sample down to county - use a spatial join for efficiency with large datasets. Sample for Uasin Gishu is 152,150 rows
sample_popclusters_df = gpd.sjoin(popclusters_df, sample_adm_df, how="inner", predicate="intersects")

popclusters_df.shape, sample_popclusters_df.shape

In [ ]:
# Create a temporary column for coloring
sample_popclusters_df["color_id"] = sample_popclusters_df['id'] % 20

In [ ]:
# Down-sample to population filter
pop_min, pop_max = 200, 5000
sample_popclusters_popfilt_df = sample_popclusters_df[sample_popclusters_df["Population"].between(pop_min, pop_max)]

sample_popclusters_df.shape, sample_popclusters_popfilt_df.shape


In [ ]:
# Explore using the transformed ID for coloring

# All
# m = sample_popclusters_df.explore(column="color_id", cmap="tab20")
# m

# Population filtered
m = sample_adm_df.explore(
    column="ADM1_EN", cmap=["red", "orange"], highlight_kwds=dict(fillOpacity=0.05),
    style_kwds=dict(opacity=0.05, fillOpacity=0.05), tiles="CartoDB positron", tooltip=False,)
m = sample_popclusters_popfilt_df.explore(m=m, column="color_id", cmap="tab20", legend=False)
m

In [ ]:
# Filter PopClusters by population range
filtered_popclusters = sample_popclusters_df[
    (sample_popclusters_df['Population'] >= 1000) &
    (sample_popclusters_df['Population'] <= 5000)
]

# Count PopClusters in each sample region
popcluster_counts_by_region = filtered_popclusters.groupby('ADM1_EN').size().reset_index(name='PopCluster_Count')

print("Number of PopClusters with population between 1000 and 5000 per region:")
print(popcluster_counts_by_region)

In [ ]:
afghana_wps_file = data_dir + "AFPW-AllPipedSystemsAndComponents.geojson"
afghana_wps_df = gpd.read_file(afghana_wps_file).to_crs(crs=PROJ_CRS)
sample_wp_df = afghana_wps_df[afghana_wps_df.apply(lambda r: r["geometry"].intersects(sample_adm_df.geometry).sum() > 0, axis=1)].copy()
afghana_wps_df.shape, sample_wp_df.shape

In [ ]:
m = sample_adm_df.explore(
    column="ADM1_EN", cmap=["red", "orange"], highlight_kwds=dict(fillOpacity=0.05),
    style_kwds=dict(opacity=0.05, fillOpacity=0.05), tiles="CartoDB positron", tooltip=False,)
m = sample_popclusters_popfilt_df.explore(
    m=m, column="color_id", cmap="tab20", legend=False)
m = sample_wp_df.explore(m=m, color="blue", marker_kwds=dict(radius=1))
m

In [ ]:
# Population hist by Urban category
fig = px.histogram(sample_popclusters_df, x="Population", facet_col="IsUrban")
fig.update_xaxes(matches=None)
fig.update_yaxes(matches=None, showticklabels=True)
fig.show()

In [ ]:
# Fine-grained hist around low populations
low_pop_df = sample_popclusters_df[sample_popclusters_df["Population"] < 100]
px.histogram(low_pop_df, x="Population")

# Explore Overlays - Comms w/WPs, Labs, PPs


`sample_hotosm_df`, `sample_wp_df`, `sample_labs_df`

In [ ]:
# Create a new DataFrame copy for the cluster proximity analysis for waterpoints
sample_wp_df_with_clusters = sample_wp_df.copy()
sample_wp_df_with_clusters['PopCluster_Membership'] = "not_in_PopCluster"

# Reuse existing buffers for PopClusters or recreate if not available from context
# (Assuming buffer_all_gdf and buffer_filtered_gdf are still in scope from previous execution)

# Spatial join with waterpoints to find intersections with all clusters
joined_wp_all_clusters = gpd.sjoin(sample_wp_df_with_clusters, buffer_all_gdf, how="inner", predicate="intersects")

# Assign 'in_PopCluster' status for waterpoints within all clusters
wp_ids_in_all_clusters = joined_wp_all_clusters.index.unique()
sample_wp_df_with_clusters.loc[wp_ids_in_all_clusters, 'PopCluster_Membership'] = 'in_PopCluster'

# Spatial join with waterpoints to find intersections with filtered clusters
joined_wp_filtered_clusters = gpd.sjoin(sample_wp_df_with_clusters, buffer_filtered_gdf, how="inner", predicate="intersects")

# Assign 'in_PopFilt_PopCluster' status for waterpoints within filtered clusters (overwrites if more specific)
wp_ids_in_filtered_clusters = joined_wp_filtered_clusters.index.unique()
sample_wp_df_with_clusters.loc[wp_ids_in_filtered_clusters, 'PopCluster_Membership'] = 'in_PopFilt_PopCluster'

In [ ]:
# Create a new DataFrame copy for the cluster proximity analysis for waterpoints
sample_wp_df_with_clusters = sample_wp_df.copy()
sample_wp_df_with_clusters['PopCluster_Membership'] = "not_in_PopCluster"

# Reuse existing buffers for PopClusters or recreate if not available from context
# (Assuming buffer_all_gdf and buffer_filtered_gdf are still in scope from previous execution)

# Spatial join with waterpoints to find intersections with all clusters
joined_wp_all_clusters = gpd.sjoin(sample_wp_df_with_clusters, buffer_all_gdf, how="inner", predicate="intersects")

# Assign 'in_PopCluster' status for waterpoints within all clusters
wp_ids_in_all_clusters = joined_wp_all_clusters.index.unique()
sample_wp_df_with_clusters.loc[wp_ids_in_all_clusters, 'PopCluster_Membership'] = 'in_PopCluster'

# Spatial join with waterpoints to find intersections with filtered clusters
joined_wp_filtered_clusters = gpd.sjoin(sample_wp_df_with_clusters, buffer_filtered_gdf, how="inner", predicate="intersects")

# Assign 'in_PopFilt_PopCluster' status for waterpoints within filtered clusters (overwrites if more specific)
wp_ids_in_filtered_clusters = joined_wp_filtered_clusters.index.unique()
sample_wp_df_with_clusters.loc[wp_ids_in_filtered_clusters, 'PopCluster_Membership'] = 'in_PopFilt_PopCluster'

In [ ]:
# Explore
m = sample_adm_df.explore(
    style_kwds=dict(color="red", opacity=0.05, fillOpacity=0.05), tiles="CartoDB positron", tooltip=False,
    highlight_kwds=dict(fillOpacity=0.05))
m = sample_popclusters_popfilt_df.explore(
    m=m, column="color_id", cmap="tab20", opacity=0.4, fillOpacity=0.4, legend=False)
m = sample_hotosm_df_with_clusters.explore(
    m=m, column="PopCluster_Membership", cmap=["green", "lime",  "red"], marker_kwds=dict(radius=5),
    legend=True, legend_kwds=dict(caption="Populated Places in PopClusters"))
m = sample_wp_df_with_clusters.explore(
    m=m, column="PopCluster_Membership", cmap=["darkblue", "deepskyblue", "orange"], marker_kwds=dict(radius=5),
    legend=True, legend_kwds=dict(caption="Waterpoints in PopClusters"))
m = sample_labs_df.explore(
    m=m, color="magenta", marker_kwds=dict(radius=5))

m

### Populated Place in PopClusters Stats

In [ ]:
# Check Populated Places against PopClusters - Popclusters w/PPs "IN" (within 500m)
buffer_filtered_clusters = sample_popclusters_popfilt_df.geometry.buffer(500)
buffer_all_clusters = sample_popclusters_df.geometry.buffer(500)

# Create a GeoDataFrame from the buffer geometries
buffer_filtered_gdf = gpd.GeoDataFrame(geometry=buffer_filtered_clusters, crs=PROJ_CRS)
buffer_all_gdf = gpd.GeoDataFrame(geometry=buffer_all_clusters, crs=PROJ_CRS)

# Spatial join with hotosm_df to find intersections
joined_filtered_clusters = gpd.sjoin(sample_hotosm_df, buffer_filtered_gdf, how="inner", predicate="intersects")
joined_all_clusters = gpd.sjoin(sample_hotosm_df, buffer_all_gdf, how="inner", predicate="intersects")

# Count unique populated places
unique_places_filtered_clusters = joined_filtered_clusters["osm_id"].nunique()
unique_places_all_clusters = joined_all_clusters["osm_id"].nunique()

# Create a new DataFrame copy for the cluster proximity analysis
sample_hotosm_df_with_clusters = sample_hotosm_df.copy()
sample_hotosm_df_with_clusters['PopCluster_Member'] = None

# Assign 'in_PopCluster' status for all clusters
# Use 'osm_id' for unique identification of populated places
ids_in_all_clusters = joined_all_clusters['osm_id'].unique()
sample_hotosm_df_with_clusters.loc[sample_hotosm_df_with_clusters['osm_id'].isin(ids_in_all_clusters), 'PopCluster_Member'] = 'in_PopCluster'

# Assign 'in_PopFilt_PopCluster' status for filtered clusters (this will overwrite if more specific)
ids_in_filtered_clusters = joined_filtered_clusters['osm_id'].unique()
sample_hotosm_df_with_clusters.loc[sample_hotosm_df_with_clusters['osm_id'].isin(ids_in_filtered_clusters), 'PopCluster_Member'] = 'in_PopFilt_PopCluster'

# --- Calculate and Print Stats ---

print("\n--- PopClusters with a Populated Place within 500m ---")

# Filtered PopClusters
num_filtered_clusters_with_pp = joined_filtered_clusters['index_right'].nunique()
total_filtered_clusters = len(sample_popclusters_popfilt_df)
percent_filtered_clusters_with_pp = (num_filtered_clusters_with_pp / total_filtered_clusters) * 100 if total_filtered_clusters > 0 else 0
print(f"Filtered PopClusters with PP: {num_filtered_clusters_with_pp} out of {total_filtered_clusters} ({percent_filtered_clusters_with_pp:.2f}%)")

# All PopClusters
num_all_clusters_with_pp = joined_all_clusters['index_right'].nunique()
total_all_clusters = len(sample_popclusters_df)
percent_all_clusters_with_pp = (num_all_clusters_with_pp / total_all_clusters) * 100 if total_all_clusters > 0 else 0
print(f"All PopClusters with PP: {num_all_clusters_with_pp} out of {total_all_clusters} ({percent_all_clusters_with_pp:.2f}%) ")

print("\n--- Populated Places within 500m of PopClusters ---")

total_hotosm_places = len(sample_hotosm_df)

# Populated Places in Filtered PopClusters
percent_pp_in_filtered_clusters = (unique_places_filtered_clusters / total_hotosm_places) * 100 if total_hotosm_places > 0 else 0
print(f"Populated Places in Filtered PopClusters: {unique_places_filtered_clusters} out of {total_hotosm_places} ({percent_pp_in_filtered_clusters:.2f}%) ")

# Populated Places in All PopClusters
percent_pp_in_all_clusters = (unique_places_all_clusters / total_hotosm_places) * 100 if total_hotosm_places > 0 else 0
print(f"Populated Places in All PopClusters: {unique_places_all_clusters} out of {total_hotosm_places} ({percent_pp_in_all_clusters:.2f}%) ")


### Waterpoints in PopClusters Stats

In [ ]:
# --- Calculate and Print Stats for Waterpoints ---
print("\n--- PopClusters with a Waterpoint within 500m ---")

# Filtered PopClusters with Waterpoints
# Use the already computed joined_wp_filtered_clusters from P7sM2kYfc0ys
num_filtered_clusters_with_wp = joined_wp_filtered_clusters['index_right'].nunique()
total_filtered_clusters = len(sample_popclusters_popfilt_df)
percent_filtered_clusters_with_wp = (num_filtered_clusters_with_wp / total_filtered_clusters) * 100 if total_filtered_clusters > 0 else 0
print(f"Filtered PopClusters with WP: {num_filtered_clusters_with_wp} out of {total_filtered_clusters} ({percent_filtered_clusters_with_wp:.2f}%)")

# All PopClusters with Waterpoints
# Use the already computed joined_wp_all_clusters from P7sM2kYfc0ys
num_all_clusters_with_wp = joined_wp_all_clusters['index_right'].nunique()
total_all_clusters = len(sample_popclusters_df)
percent_all_clusters_with_wp = (num_all_clusters_with_wp / total_all_clusters) * 100 if total_all_clusters > 0 else 0
print(f"All PopClusters with WP: {num_all_clusters_with_wp} out of {total_all_clusters} ({percent_all_clusters_with_wp:.2f}%)\n")

print("--- Waterpoints within 500m of PopClusters ---")

total_waterpoints = len(sample_wp_df)

# Waterpoints in Filtered PopClusters (already calculated in P7sM2kYfc0ys)
wp_ids_in_filtered_clusters = joined_wp_filtered_clusters.index.unique()
unique_wp_in_filtered_clusters = len(wp_ids_in_filtered_clusters)
percent_wp_in_filtered_clusters = (unique_wp_in_filtered_clusters / total_waterpoints) * 100 if total_waterpoints > 0 else 0
print(f"Waterpoints in Filtered PopClusters: {unique_wp_in_filtered_clusters} out of {total_waterpoints} ({percent_wp_in_filtered_clusters:.2f}%)")

# Waterpoints in All PopClusters (already calculated in P7sM2kYfc0ys)
wp_ids_in_all_clusters = joined_wp_all_clusters.index.unique()
unique_wp_in_all_clusters = len(wp_ids_in_all_clusters)
percent_wp_in_all_clusters = (unique_wp_in_all_clusters / total_waterpoints) * 100 if total_waterpoints > 0 else 0
print(f"Waterpoints in All PopClusters: {unique_wp_in_all_clusters} out of {total_waterpoints} ({percent_wp_in_all_clusters:.2f}%)")

In [ ]:
# Identify unique PopClusters (from sample_popclusters_df) that have a waterpoint within 500m
popclusters_with_wp_indices = joined_wp_all_clusters['index_right'].unique()
popclusters_with_wp_data = sample_popclusters_df.loc[popclusters_with_wp_indices]

# Sub filter for better vis
popclusters_with_wp_data = popclusters_with_wp_data[popclusters_with_wp_data["Population"] < 2100]

# Create a cumulative distribution plot of the population of these PopClusters
fig = px.histogram(
    popclusters_with_wp_data,
    x="Population",
    title="Cumulative Distribution of Population for PopClusters WITH Waterpoints within 500m",
    histnorm='percent',
    cumulative=True) # Set cumulative to True for cumulative distribution

fig.update_xaxes(title_text="Population") # Add x-axis limit
fig.update_yaxes(title_text="Cumulative Percentage of PopClusters") # Update Y-axis title for cumulative
fig.update_layout(bargap=0.2, width=1000) # Add spacing between bars and set width

# Add vertical red line at x=200
fig.add_vline(x=200, line_color="red", line_dash="dash", annotation_text="Pop=200", annotation_position="top right")

fig.show()

In [ ]:
import plotly.graph_objects as go

# Identify unique PopFilt PopClusters (from sample_popclusters_popfilt_df) that have a waterpoint within 500m
# Use 'index_right' to get the indices of the popclusters from the joined_wp_filtered_clusters
popfilt_popclusters_with_wp_indices = joined_wp_filtered_clusters['index_right'].unique()
popfilt_popclusters_with_wp_data = sample_popclusters_popfilt_df.loc[popfilt_popclusters_with_wp_indices]

# Get min and max population from the filtered data
min_pop_data = popfilt_popclusters_with_wp_data['Population'].min()
max_pop_data = popfilt_popclusters_with_wp_data['Population'].max()

# Use the globally defined pop_min and pop_max for bin calculation as they represent the filter range
# We want the bins to start at pop_min (200) and end at pop_max (5000)
bin_start = pop_min
bin_end = pop_max
num_bins_desired = 50
bin_size = 100

# Create a go.Figure and add a go.Histogram trace
fig = go.Figure(data=[go.Histogram(
    x=popfilt_popclusters_with_wp_data['Population'],
    xbins=dict(
        start=bin_start,
        end=bin_end,
        size=bin_size
    ),
    marker_line_width=1,
    marker_line_color='white'
)])

# Update layout for title, axis labels, bargap, and width
fig.update_layout(
    title="Population of PopFilt PopClusters WITH Waterpoints within 500m",
    xaxis_title="Population",
    yaxis_title="Count of PopFilt PopClusters",
    bargap=0.2,
    width=1000
)

# Add granular ticks and tilt labels
fig.update_xaxes(tickmode='linear', dtick=100, tickangle=-45)

fig.show()

In [ ]:
# Identify PopClusters with waterpoints within 500m (reusing previous logic)
popclusters_with_wp_indices = joined_wp_all_clusters['index_right'].unique()
popclusters_with_wp = sample_popclusters_df.loc[popclusters_with_wp_indices].copy()

# Identify PopClusters without waterpoints within 500m
popclusters_without_wp = sample_popclusters_df[~sample_popclusters_df.index.isin(popclusters_with_wp_indices)].copy()

# Add a 'Status' column to each DataFrame for differentiation
popclusters_with_wp['Status'] = 'With Waterpoint'
popclusters_without_wp['Status'] = 'Without Waterpoint'

# Concatenate the two DataFrames
combined_popclusters_df = pd.concat([popclusters_with_wp, popclusters_without_wp])

# Filter the combined DataFrame to the desired population range (0-2100)
combined_popclusters_filtered = combined_popclusters_df[combined_popclusters_df['Population'] <= 2100].copy()

# Create the combined histogram
fig = px.histogram(
    combined_popclusters_filtered,
    x="Population",
    color="Status",
    barmode="overlay",
    histnorm="percent",
    opacity=0.5,
    title="Population Distribution of PopClusters (With vs. Without Waterpoints within 500m)",
    nbins=50,
    #category_orders={"Status": ["Without Waterpoint", "With Waterpoint"]} # Set stacking order directly in px.histogram
)

fig.update_xaxes(title_text="Population", range=[0, 2100]) # Set x-axis display range
fig.update_yaxes(title_text="Percentage of PopClusters")
fig.update_layout(bargap=0.1, width=1000)

fig.show()

# Consider PopCluster Merging

(Or re-"clustering", as in Saha approach) to get a better picture of actual communities.

Can then look at where WPs land; make more informed judgements about counts.